# Monte Carlo portfolio simulation

Load the portfolio config and historical prices, then project the portfolio forward with a block bootstrap of historical monthly returns.


In [ ]:
import pandas as pd

from portfolio_lab.analysis import (
    annualised_stats,
    contributed_path,
    contributions_by_asset,
    final_value_summary,
    gain_by_asset,
    gain_summary,
    monthly_returns_from_prices,
    plot_fan_chart,
    plot_final_histogram,
    plot_gain_fan_chart,
    plot_weights_over_time,
    run_simulation,
    save_figure,
    weights_summary,
)
from portfolio_lab.config import load_config
from portfolio_lab.data import load_prices

## Configuration

Assets and simulation settings from `config/portfolio.toml` (falls back to the example file if missing).


In [ ]:
cfg = load_config()

display(pd.DataFrame([a.model_dump() for a in cfg.portfolio.assets]))
print(cfg.portfolio.name, cfg.portfolio.currency)
print(cfg.simulation)

## Historical data

Daily prices for every asset, cached in `data/raw/`, converted to monthly returns. Only dates where all assets have data are kept, so the shortest history sets the window.


In [ ]:
tickers = [a.ticker for a in cfg.portfolio.assets]
prices = load_prices(tickers)
returns = monthly_returns_from_prices(prices, tickers)

print(f"{returns.index[0]:%Y-%m} to {returns.index[-1]:%Y-%m} ({len(returns)} months)")
annualised_stats(returns).round(3)

## Simulation

Block bootstrap: each path stitches together 12-month blocks of historical returns. The monthly contribution is split by `contribution_weight` and added at the start of each month.


In [ ]:
paths = run_simulation(cfg, returns)  # (n_paths, n_months + 1, n_assets)
total = paths.sum(axis=2)
contributed = contributed_path(cfg, start="initial_value")
cost = contributed_path(cfg, start="cost_basis")
total.shape

## Results


In [ ]:
summary = final_value_summary(total, contributed, cost)
summary.to_frame(f"Final value ({cfg.portfolio.currency})").map("{:,.2f}".format)

In [ ]:
currency = cfg.portfolio.currency
fan = plot_fan_chart(total, contributed, currency, cost=cost)
fan_log = plot_fan_chart(total, contributed, currency, log_scale=True, cost=cost)
save_figure(fan, "fan_chart")
save_figure(fan_log, "fan_chart_log");

In [ ]:
hist = plot_final_histogram(total, contributed, cfg.portfolio.currency, cost=cost)
save_figure(hist, "final_value_histogram");

### Asset weights over time

How the portfolio weights evolve over time. If the portfolio is rebalanced, the weights will converge to the target weights. If not, they will drift with the relative performance of each asset.


In [ ]:
asset_names = [a.name for a in cfg.portfolio.assets]
display(weights_summary(paths, asset_names).map("{:.1%}".format))

weights_fig = plot_weights_over_time(paths, asset_names)
save_figure(weights_fig, "asset_weights");

## Gain vs cost basis

How much the portfolio is worth compared with what was actually paid: `cost_basis` per asset plus the monthly contributions. Gain = simulated value − (cost basis + contributions).


In [ ]:
gains = gain_summary(total, cost)
gains.map(lambda v: f"{v:,.2f}").assign(Return=gains["Return"].map("{:.1%}".format))

In [ ]:
gain_fig = plot_gain_fan_chart(total, cost, cfg.portfolio.currency)
save_figure(gain_fig, "gain_fan_chart");

### Per asset

Final gain for each asset against its own cost basis plus its share of the contributions.


In [ ]:
by_asset = gain_by_asset(
    paths, contributions_by_asset(cfg, "cost_basis"), [a.name for a in cfg.portfolio.assets]
)

formatted_by_asset = pd.DataFrame(
    {
        col: values.map("{:.1%}".format if col == "P50 return" else "{:,.2f}".format)
        for col, values in by_asset.items()
    }
)

formatted_by_asset